# Coordinate reset: training SIBI/BISINDO di Colab

Pilih **Runtime ? Change runtime type ? GPU**. Upload paket koordinat yang dibuat di komputer lokal dengan `python scripts/package_coordinates_colab.py --language sibi`.

Notebook melatih CNN1D dan MLP, lalu membandingkan Softmax dan SVM pada embedding. Tidak memuat foto atau test arrays. J/Z SIBI adalah pose statis dari ASL, bukan pengenal gerakan.


In [ ]:
%pip install -q "scikit-learn==1.5.2" "joblib==1.4.2" "matplotlib>=3.8,<4"
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback; aktifkan GPU bila diinginkan')


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os
uploaded = files.upload()  # coordinate-reset-v1-sibi-colab.zip
if len(uploaded) != 1:
    raise ValueError('Upload satu paket ZIP koordinat saja')
project = Path('/content/coordinate-project')
project.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for item in archive.infolist():
        target = (project / item.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(project)
os.chdir(project)
print('Project:', project)


In [ ]:
# Pilih sibi atau bisindo sesuai paket yang diupload.
LANGUAGE = 'sibi'
MODELS = ['cnn', 'mlp']  # ['cnn'] untuk fokus CNN saja
EPOCHS = 100
RUN_DIR = f'artifacts/coordinate-reset-v1/{LANGUAGE}_colab'

import subprocess, sys
command = [sys.executable, '-u', 'scripts/train_coordinates.py', '--language', LANGUAGE,
           '--models', *MODELS, '--epochs', str(EPOCHS), '--output-dir', RUN_DIR]
# Output per epoch langsung tampil; hentikan jika konfigurasi salah.
subprocess.run(command, check=True)


In [ ]:
import json
from IPython.display import display, Image
selection = json.loads((Path(RUN_DIR) / 'selection.json').read_text())
print('Selected:', selection['selected'])
print('Test loaded:', selection['test_loaded'])
display(Image(filename=str(Path(RUN_DIR) / 'learning_curves.png')))
# Detail per kelas/sumber ada di *_validation.json; epoch ada di *_epochs.csv.


In [ ]:
import zipfile
archive_path = Path('/content') / f'{LANGUAGE}-coordinate-reset-results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in Path(RUN_DIR).rglob('*'):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(archive_path))


## Kembali ke komputer lokal

Ekstrak ZIP hasil ke akar proyek dengan mempertahankan folder `artifacts/`. Jalankan:

```powershell
python scripts/webcam_coordinates.py --run-dir artifacts/coordinate-reset-v1/sibi_colab --check-only
python scripts/webcam_coordinates.py --run-dir artifacts/coordinate-reset-v1/sibi_colab
```

Gunakan environment MediaPipe lokal lama untuk webcam. Skor Softmax/margin SVM belum terkalibrasi; performa validation tidak menjamin performa webcam. Source notebook ini tidak menyimpan output training.
